# OmniScore Evaluation on XSAMSum (English → Chinese)

This notebook evaluates Chinese summaries on part of the XSAMSum dataset using
**OmniScore** (Alam, Bhatia, Laskar, Chowdhury 2026), a deterministic encoder-based
multilingual evaluator that scores generated text on four dimensions:
**informativeness**, **clarity**, **plausibility**, **faithfulness**.
Scores are continuous in `[1, 5]`.

We use the [`QCRI/OmniScore-deberta-v3`](https://huggingface.co/QCRI/OmniScore-deberta-v3)
checkpoint (0.2B parameters, DeBERTa-v3-base backbone, custom regression head).

## What this notebook does

The input is a single CSV that may contain results from one or more models,
distinguished by the `model_name` column. Each row contains:

- `dialogue` — English source dialogue (the source)
- `reference_summary_zh` — Chinese gold-reference summary
- `generated_summary_zh` — Chinese candidate summary produced by the model

We score each candidate in **two modes**:

| Mode | Inputs to OmniScore | What it tells us |
|------|---------------------|------------------|
| Source-grounded | `Source = English dialogue`, `Candidate = Chinese summary` | Faithfulness to the English source (catches cross-lingual hallucination) |
| Reference-based | `Reference = Chinese gold`, `Candidate = Chinese summary` | Closeness to the human-written Chinese summary |

For each mode we get four scalar scores per example, then aggregate to corpus level — **separately for each model**.

## Layout

1. Setup & dependencies
2. Load OmniScore
3. Load the results CSV and group by model
4. Format inputs for OmniScore
5. Run scoring (both modes, batched, per model)
6. Aggregate and report (per model)
7. Save results to disk

## Important caveats

- OmniScore was trained on 107 languages but Chinese-specific correlation
  numbers are not separately reported in the paper. Treat these scores as a
  *signal*, not ground truth.
- The model truncates inputs at 512 tokens. SAMSum dialogues can be longer
  than that once formatted; we report truncation rates so you know how much
  of each example actually gets scored.
- These scores are not directly comparable to ROUGE/BERTScore or to ClidSum's
  human-study axes (grammaticality / informativeness / conciseness).
  OmniScore extends the evaluation; it does not replace it.

## 1. Setup & dependencies

In [ ]:
!pip uninstall -y torchvision torchaudio

In [ ]:
# Install required packages (uncomment if running fresh)
!pip install -q -U torch transformers sentencepiece datasets pandas tqdm

In [ ]:
import os
import json
import math
import time
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

# Reproducibility — OmniScore is deterministic at inference, but we still
# fix seeds so that any sampling we do (e.g. selecting a slice) is stable.
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 2. Load OmniScore

The OmniScore checkpoint ships a custom `ScorePredictorModel` class, so we need
`trust_remote_code=True`. The score names and output range come from the model
config — we don't hardcode them.

In [ ]:
REPO_ID = "QCRI/OmniScore-deberta-v3"
MAX_LEN = 512  # the model's own max sequence length

tokenizer = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(REPO_ID, trust_remote_code=True).to(DEVICE).eval()

SCORE_NAMES = list(model.config.score_names)
print("Score dimensions:", SCORE_NAMES)
print("Output range: [1, 5] (sigmoid-scaled in the regression head)")

## 3. Load the results CSV and group by model

The input CSV may contain results from multiple models, distinguished by the
`model_name` column. The columns we need:

- `dialogue` — English source dialogue
- `reference_summary_zh` — Chinese gold-reference summary
- `generated_summary_zh` — Chinese candidate summary

If a `test_index` column isn't present, the row index is used as the example id.
Metrics are computed independently for each model.

In [ ]:
INPUT_CSV = "results.csv"  # path to the CSV containing generated + reference summaries for one or more models

MODEL_COL = "model_name"
PRED_COL = "generated_summary_zh"
REF_COL = "reference_summary_zh"
DIALOGUE_COL = "dialogue"

df = pd.read_csv(INPUT_CSV)

required_cols = {MODEL_COL, PRED_COL, REF_COL, DIALOGUE_COL}
missing = required_cols - set(df.columns)
assert not missing, f"Missing expected columns: {missing}"

# Drop rows missing any of the fields we need
df = df.dropna(subset=[PRED_COL, REF_COL, DIALOGUE_COL]).reset_index(drop=True)

if "test_index" not in df.columns:
    df["test_index"] = df.index.astype(str)

groups = {
    name: g.reset_index(drop=True)
    for name, g in df.groupby(MODEL_COL)
}

for name, g in groups.items():
    print(f"{name}: {len(g)} examples")

## 4. Format inputs for OmniScore

OmniScore expects a **single flat text string** following this schema (from the
official model card):

```
Task: <task_name>
Source: <source text, if available>
Reference: <reference text, if available>
Candidate: <model output being evaluated>
```

For our two scoring modes:

- **Source-grounded**: include `Task` + `Source` (English dialogue) + `Candidate` (Chinese summary). Omit `Reference`.
- **Reference-based**: include `Task` + `Reference` (Chinese gold) + `Candidate` (Chinese summary). Omit `Source`.

We use `Task: summarization` since the candidate is a summary; the model card
lists `Summarization evaluation` as one of the supported tasks.

The 512-token limit will bite on long English dialogues. We don't manually
truncate — we let the tokenizer do it (`truncation=True`) but record how often
truncation happens so it's reported alongside the scores.

In [ ]:
TASK_TAG = "summarization"

def format_source_grounded(row):
    return (
        f"Task: {TASK_TAG}\n"
        f"Source: {row[DIALOGUE_COL]}\n"
        f"Candidate: {row[PRED_COL]}"
    )

def format_reference_based(row):
    return (
        f"Task: {TASK_TAG}\n"
        f"Reference: {row[REF_COL]}\n"
        f"Candidate: {row[PRED_COL]}"
    )

# Sanity check on the first example of the first model
_first_name = next(iter(groups))
_first_row = groups[_first_name].iloc[0]

print(f"=== Source-grounded input ({_first_name}) ===")
print(format_source_grounded(_first_row)[:400], "...")
print()
print(f"=== Reference-based input ({_first_name}) ===")
print(format_reference_based(_first_row)[:400], "...")

## 5. Run scoring (batched, per model)

We score in mini-batches and:

1. Track per-example truncation (was the input clipped at 512 tokens?).
2. Run the model with `torch.no_grad()` and `.eval()` mode (already set).
3. Pull predictions from `outputs.predictions`, which has shape
   `(batch, n_score_dims)` and lives in `[1, 5]`.
4. Map each column back to its score name using `model.config.score_names`.

Both modes are run for every model in `groups`, and the results are tagged
with `model_name` so they can be aggregated separately.

In [ ]:
BATCH_SIZE = 8  # safe default for a 0.2B model on a single GPU; tune for your hardware

def _was_truncated(text):
    # Did the tokenizer cut this input at MAX_LEN?
    ids = tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"]
    return len(ids) > MAX_LEN

def score_batch(texts):
    # Score a list of inputs. Returns (scores_per_input, truncation_flags).
    trunc_flags = [_was_truncated(t) for t in texts]
    batch = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    with torch.no_grad():
        out = model(**batch)
    preds = out.predictions.detach().cpu()  # shape (B, len(SCORE_NAMES))
    rows = []
    for row in preds:
        rows.append({name: float(row[i]) for i, name in enumerate(SCORE_NAMES)})
    return rows, trunc_flags

def score_all(records, formatter, mode_name, desc=""):
    # records: list of dicts (rows), each with at least test_index + the fields formatter needs
    all_rows = []
    n = len(records)
    t0 = time.time()
    for start in tqdm(range(0, n, BATCH_SIZE), desc=f"OmniScore [{desc}/{mode_name}]"):
        chunk = records[start : start + BATCH_SIZE]
        texts = [formatter(ex) for ex in chunk]
        scores, trunc_flags = score_batch(texts)
        for ex, sc, tr in zip(chunk, scores, trunc_flags):
            row = {"example_id": ex["test_index"], "mode": mode_name, "truncated": tr}
            row.update(sc)
            all_rows.append(row)
    elapsed = time.time() - t0
    print(f"[{desc}/{mode_name}] scored {n} examples in {elapsed:.1f}s "
          f"({n / max(elapsed, 1e-9):.1f} ex/s)")
    return pd.DataFrame(all_rows)

all_dfs = []
for name, g in groups.items():
    records = g.to_dict("records")
    df_source_m = score_all(records, format_source_grounded, "source_grounded", desc=name)
    df_ref_m    = score_all(records, format_reference_based, "reference_based", desc=name)
    df_model = pd.concat([df_source_m, df_ref_m], ignore_index=True)
    df_model[MODEL_COL] = name
    all_dfs.append(df_model)

df_all = pd.concat(all_dfs, ignore_index=True)
df_all

## 6. Aggregate and report

Corpus-level numbers: mean and standard deviation per score dimension, per mode,
per model, plus the share of examples that were truncated. Since OmniScore outputs
are in `[1, 5]`, a useful sanity floor for the candidate quality is around 3 —
anything substantially below that on faithfulness for source-grounded scoring is a
red flag for hallucination.

In [ ]:
def summarize(df):
    g = df.groupby([MODEL_COL, "mode"])
    means = g[SCORE_NAMES].mean().add_suffix("_mean")
    stds  = g[SCORE_NAMES].std().add_suffix("_std")
    trunc = g["truncated"].mean().rename("truncation_rate").to_frame()
    count = g.size().rename("n").to_frame()
    out = pd.concat([count, trunc, means, stds], axis=1)
    # Reorder: count, trunc, then mean/std interleaved per dim
    cols = ["n", "truncation_rate"]
    for s in SCORE_NAMES:
        cols.extend([f"{s}_mean", f"{s}_std"])
    return out[cols]

summary = summarize(df_all)
summary

In [ ]:
# Per-example view — easier to inspect outliers (e.g. low faithfulness scores)
view = df_all.pivot_table(
    index=[MODEL_COL, "example_id"],
    columns="mode",
    values=SCORE_NAMES,
    aggfunc="first",
)
# Flatten the column MultiIndex: (score, mode) -> "score__mode"
view.columns = [f"{s}__{m}" for s, m in view.columns]
view = view.reset_index()
view

In [ ]:
# Flag examples worth eyeballing: low faithfulness in source-grounded mode
LOW_FAITH_THRESHOLD = 3.0
df_source_all = df_all[df_all["mode"] == "source_grounded"]
flagged = df_source_all[df_source_all["faithfulness"] < LOW_FAITH_THRESHOLD].copy()
if len(flagged):
    print(f"{len(flagged)} example(s) below faithfulness {LOW_FAITH_THRESHOLD} "
          f"in source-grounded mode — likely hallucination candidates:")
    print(flagged[[MODEL_COL, "example_id", "faithfulness", "informativeness", "clarity", "plausibility"]])
else:
    print(f"No examples below faithfulness {LOW_FAITH_THRESHOLD} in source-grounded mode.")

## 7. Save results

We dump three artifacts:

- `omniscore_per_example.csv` — every score for every example in both modes
- `omniscore_summary.csv` — corpus-level means/stds per mode
- `omniscore_run_meta.json` — model id, n examples, device, score dimensions, timestamp

Together these make the run reproducible and easy to drop into a later
comparison table (alongside ROUGE / BERTScore / human study numbers).

In [ ]:
from datetime import datetime, timezone

OUT_DIR = Path("./omniscore_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

per_example_path = OUT_DIR / "omniscore_per_example.csv"
summary_path     = OUT_DIR / "omniscore_summary.csv"
meta_path        = OUT_DIR / "omniscore_run_meta.json"

df_all.to_csv(per_example_path, index=False, encoding="utf-8")
summary.to_csv(summary_path, encoding="utf-8")

meta = {
    "model": REPO_ID,
    "max_seq_len": MAX_LEN,
    "score_names": SCORE_NAMES,
    "score_range": [1.0, 5.0],
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "input_csv": INPUT_CSV,
    "model_names": list(groups.keys()),
    "n_examples_per_model": {name: len(g) for name, g in groups.items()},
    "modes": ["source_grounded", "reference_based"],
    "task_tag": TASK_TAG,
    "seed": SEED,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

print("Wrote:")
for p in (per_example_path, summary_path, meta_path):
    print(f"  {p}  ({p.stat().st_size} bytes)")